# Face 3D Reconstruction with VGGT

This notebook uses [VGGT (Visual Geometry Grounded Transformer)](https://github.com/facebookresearch/vggt), a CVPR 2025 Best Paper, to reconstruct a 3D point cloud of a face from multiple images.

**Pipeline:**
1. Install dependencies & clone VGGT
2. Upload your face images
3. Run VGGT inference to get camera poses, depth maps, and 3D points
4. Visualize the 3D point cloud interactively
5. Export as a `.ply` file for use in Blender, MeshLab, etc.

> **Runtime:** Make sure you're using a GPU runtime: `Runtime > Change runtime type > T4 GPU`

## 1. Setup: Install Dependencies & Clone VGGT

In [ ]:
import os

# Clone VGGT if not already present
if not os.path.exists('vggt'):
    !git clone https://github.com/facebookresearch/vggt.git

%cd vggt

# Install core dependencies
!pip install -q einops safetensors huggingface_hub

# Install plotly for interactive 3D visualization
!pip install -q plotly

print("Setup complete!")

In [ ]:
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import glob
import os

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Inference will be very slow on CPU.")

## 2. Upload Face Images

Choose **one** of the two options below:
- **Option A**: Upload images directly (simple, up to ~20 images)
- **Option B**: Mount Google Drive (better for larger sets)

Images should be of the same person from multiple angles (front, side, 3/4 view, etc.). More angles = better reconstruction.

In [ ]:
# ============================================================
# OPTION A: Direct upload
# ============================================================
from google.colab import files

os.makedirs('/content/face_images', exist_ok=True)

print("Select your face images to upload (PNG, JPG, JPEG supported):")
uploaded = files.upload()

for filename, data in uploaded.items():
    filepath = f'/content/face_images/{filename}'
    with open(filepath, 'wb') as f:
        f.write(data)
    print(f"  Saved: {filepath}")

IMAGE_FOLDER = '/content/face_images'
print(f"\nImages saved to: {IMAGE_FOLDER}")

In [ ]:
# ============================================================
# OPTION B: Mount Google Drive
# ============================================================
# Uncomment and run this cell instead of Option A

# from google.colab import drive
# drive.mount('/content/drive')

# # Update this path to your images folder in Drive
# IMAGE_FOLDER = '/content/drive/MyDrive/face_images'
# print(f"Using images from: {IMAGE_FOLDER}")

In [ ]:
# Find all images in the folder
image_extensions = ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG')
image_paths = []
for ext in image_extensions:
    image_paths.extend(glob.glob(os.path.join(IMAGE_FOLDER, ext)))
image_paths = sorted(image_paths)

print(f"Found {len(image_paths)} images:")
for p in image_paths:
    print(f"  {os.path.basename(p)}")

if len(image_paths) == 0:
    raise ValueError(f"No images found in {IMAGE_FOLDER}. Please upload images first.")
if len(image_paths) < 3:
    print("\nWARNING: For best results, use at least 5-10 images from different angles.")

In [ ]:
# Preview the uploaded images
n_preview = min(len(image_paths), 12)
cols = min(4, n_preview)
rows = (n_preview + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = np.array(axes).flatten() if n_preview > 1 else [axes]

for i, path in enumerate(image_paths[:n_preview]):
    img = Image.open(path).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(os.path.basename(path)[:20], fontsize=8)
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'Input Images ({len(image_paths)} total)', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Load VGGT Model

In [ ]:
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.geometry import closed_form_inverse_se3, unproject_depth_map_to_point_map
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

print("Loading VGGT-1B model from Hugging Face...")
print("(First run will download ~4GB weights — this may take a few minutes)")

model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
model.eval()

print("\nModel loaded successfully!")

## 4. Preprocess Images & Run Inference

In [ ]:
print(f"Preprocessing {len(image_paths)} images...")
# 'pad' mode preserves full content — better for face images than 'crop'
images = load_and_preprocess_images(image_paths, mode='pad').to(device)
print(f"Preprocessed image tensor shape: {images.shape}  (N, C, H, W)")

In [ ]:
# Choose dtype based on GPU capability
# bfloat16 requires Ampere+ (A100, 3090, etc.); float16 works on T4
if device == "cuda":
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    dtype = torch.float32

print(f"Running inference with dtype={dtype}...")

with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype, enabled=(device == "cuda")):
        predictions = model(images)

print("Inference complete!")
print("Prediction keys:", list(predictions.keys()))

In [ ]:
# Decode pose encoding into extrinsic (world-to-camera) and intrinsic matrices
print("Decoding camera parameters...")
extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions["pose_enc"], images.shape[-2:])
predictions["extrinsic"] = extrinsic
predictions["intrinsic"] = intrinsic

# Move everything to CPU numpy (remove batch dimension)
preds = {}
for key, val in predictions.items():
    if isinstance(val, torch.Tensor):
        preds[key] = val.float().cpu().numpy().squeeze(0)
    else:
        preds[key] = val

print("Output shapes:")
for k, v in preds.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: {v.shape}")

## 5. Visualize Depth Maps

In [ ]:
n_show = min(len(image_paths), 6)
fig, axes = plt.subplots(2, n_show, figsize=(n_show * 3, 6))

images_np = images.float().cpu().numpy()  # (N, 3, H, W)
depth_maps = preds['depth']              # (N, H, W, 1)
depth_conf = preds['depth_conf']         # (N, H, W)

for i in range(n_show):
    # Input image
    img = images_np[i].transpose(1, 2, 0)  # (H, W, 3)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Image {i+1}', fontsize=9)
    axes[0, i].axis('off')

    # Depth map (confidence-weighted for better visualization)
    depth = depth_maps[i, :, :, 0]
    conf = depth_conf[i]
    # Mask low-confidence regions
    masked_depth = np.where(conf > np.percentile(conf, 20), depth, np.nan)
    axes[1, i].imshow(masked_depth, cmap='plasma')
    axes[1, i].set_title(f'Depth {i+1}', fontsize=9)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Input', fontsize=10)
axes[1, 0].set_ylabel('Depth Map', fontsize=10)
plt.suptitle('Input Images vs Predicted Depth Maps', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Build 3D Point Cloud

In [ ]:
# Reconstruct 3D points by unprojecting depth maps through estimated cameras.
# This typically gives more accurate results than the direct point map branch.

extrinsics = preds['extrinsic']   # (N, 3, 4) — world-to-camera
intrinsics = preds['intrinsic']   # (N, 3, 3)

depth_tensor = torch.from_numpy(preds['depth'])        # (N, H, W, 1)
extrinsic_tensor = torch.from_numpy(extrinsics)        # (N, 3, 4)
intrinsic_tensor = torch.from_numpy(intrinsics)        # (N, 3, 3)

world_points = unproject_depth_map_to_point_map(
    depth_tensor, extrinsic_tensor, intrinsic_tensor
).numpy()  # (N, H, W, 3)

conf = preds['depth_conf']   # (N, H, W)

print(f"World points shape: {world_points.shape}")
print(f"Confidence shape:   {conf.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Filter by confidence and assemble colored point cloud
# ---------------------------------------------------------------------------

# Confidence percentile threshold — raise to keep fewer, higher-quality points
CONF_PERCENTILE = 40  # keep top 60% of points by confidence

conf_flat = conf.reshape(-1)
threshold = np.percentile(conf_flat, CONF_PERCENTILE)
mask = (conf_flat >= threshold) & (conf_flat > 1e-5)

# Colors from input images
colors_np = images_np.transpose(0, 2, 3, 1)  # (N, H, W, 3)
colors_flat = (colors_np.reshape(-1, 3) * 255).astype(np.uint8)
points_flat = world_points.reshape(-1, 3)

pts = points_flat[mask]
clrs = colors_flat[mask]

# Center the point cloud
center = pts.mean(axis=0)
pts_centered = pts - center

print(f"Total points after filtering: {len(pts):,}  (threshold percentile: {CONF_PERCENTILE})")

## 7. Interactive 3D Visualization

In [ ]:
import plotly.graph_objects as go

# Subsample for interactive performance (plotly handles ~200k points well in Colab)
MAX_DISPLAY_POINTS = 150_000
if len(pts_centered) > MAX_DISPLAY_POINTS:
    idx = np.random.choice(len(pts_centered), MAX_DISPLAY_POINTS, replace=False)
    pts_display = pts_centered[idx]
    clrs_display = clrs[idx]
    print(f"Subsampled to {MAX_DISPLAY_POINTS:,} points for display")
else:
    pts_display = pts_centered
    clrs_display = clrs

point_colors = [f'rgb({r},{g},{b})' for r, g, b in clrs_display]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=pts_display[:, 0],
            y=pts_display[:, 1],
            z=pts_display[:, 2],
            mode='markers',
            marker=dict(
                size=1.2,
                color=point_colors,
                opacity=0.9,
            ),
        )
    ]
)

fig.update_layout(
    title='3D Face Reconstruction (VGGT)',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data',
        bgcolor='black',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    ),
    paper_bgcolor='black',
    font_color='white',
    margin=dict(l=0, r=0, t=40, b=0),
    height=700,
)

fig.show()

## 8. (Optional) Adjust Confidence Threshold

If the reconstruction looks noisy, raise the threshold. If it looks sparse, lower it.

In [ ]:
# -----------------------------------------------
# Tune this value and re-run to adjust quality
# -----------------------------------------------
CONF_PERCENTILE = 60   # 0 = keep all points, 80 = keep only top 20%

threshold = np.percentile(conf_flat, CONF_PERCENTILE)
mask = (conf_flat >= threshold) & (conf_flat > 1e-5)

pts = points_flat[mask]
clrs = colors_flat[mask]
pts_centered = pts - pts.mean(axis=0)

print(f"Points after filtering: {len(pts):,}")

# Re-plot
if len(pts_centered) > MAX_DISPLAY_POINTS:
    idx = np.random.choice(len(pts_centered), MAX_DISPLAY_POINTS, replace=False)
    pts_display = pts_centered[idx]
    clrs_display = clrs[idx]
else:
    pts_display = pts_centered
    clrs_display = clrs

point_colors = [f'rgb({r},{g},{b})' for r, g, b in clrs_display]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=pts_display[:, 0],
            y=pts_display[:, 1],
            z=pts_display[:, 2],
            mode='markers',
            marker=dict(size=1.2, color=point_colors, opacity=0.9),
        )
    ]
)
fig.update_layout(
    title=f'3D Face Reconstruction — conf percentile={CONF_PERCENTILE}',
    scene=dict(aspectmode='data', bgcolor='black',
               xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
               yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
               zaxis=dict(showgrid=False, zeroline=False, showticklabels=False)),
    paper_bgcolor='black', font_color='white',
    margin=dict(l=0, r=0, t=40, b=0), height=700,
)
fig.show()

## 9. Export 3D Point Cloud as PLY

The `.ply` file can be opened in **Blender**, **MeshLab**, **CloudCompare**, or any 3D viewer.

In [ ]:
def save_ply(filepath, points, colors):
    """Save a colored point cloud to a PLY file."""
    assert points.shape[0] == colors.shape[0], "Points and colors must have same length"
    n = len(points)

    header = (
        f"ply\n"
        f"format ascii 1.0\n"
        f"element vertex {n}\n"
        f"property float x\n"
        f"property float y\n"
        f"property float z\n"
        f"property uchar red\n"
        f"property uchar green\n"
        f"property uchar blue\n"
        f"end_header\n"
    )

    with open(filepath, 'w') as f:
        f.write(header)
        for i in range(n):
            x, y, z = points[i]
            r, g, b = colors[i]
            f.write(f"{x:.6f} {y:.6f} {z:.6f} {int(r)} {int(g)} {int(b)}\n")

    print(f"Saved {n:,} points to {filepath}")


PLY_PATH = '/content/face_reconstruction.ply'
save_ply(PLY_PATH, pts_centered, clrs)

In [ ]:
# Download the PLY file
from google.colab import files
files.download(PLY_PATH)
print("Download started!")

## 10. (Optional) Export to COLMAP Format

COLMAP format enables further processing with Gaussian Splatting tools like [gsplat](https://github.com/nerfstudio-project/gsplat) or [Nerfstudio](https://github.com/nerfstudio-project/nerfstudio).

In [ ]:
# Install pycolmap for COLMAP export
!pip install -q pycolmap

# Arrange images in the structure demo_colmap.py expects:
# /content/scene/images/  <-- put your images here
import shutil

SCENE_DIR = '/content/scene'
SCENE_IMAGES_DIR = os.path.join(SCENE_DIR, 'images')
os.makedirs(SCENE_IMAGES_DIR, exist_ok=True)

for p in image_paths:
    shutil.copy(p, SCENE_IMAGES_DIR)

print(f"Copied {len(image_paths)} images to {SCENE_IMAGES_DIR}")
print("\nRunning COLMAP export (feedforward only, no bundle adjustment)...")
!python demo_colmap.py --scene_dir={SCENE_DIR}

print("\nCOLMAP output written to:", os.path.join(SCENE_DIR, 'sparse'))

In [ ]:
# Zip and download the COLMAP output
import shutil
shutil.make_archive('/content/colmap_output', 'zip', SCENE_DIR)
files.download('/content/colmap_output.zip')
print("COLMAP archive download started!")

---
## Tips for Better Results

| Issue | Fix |
|-------|-----|
| Reconstruction is sparse | Lower `CONF_PERCENTILE` (e.g., 20–40) |
| Too much noise / floating points | Raise `CONF_PERCENTILE` (e.g., 60–80) |
| Background included | Crop images to just the face before uploading |
| Missing parts of the face | Add more images covering those angles |
| OOM on T4 GPU | Reduce image count to ≤10, or resize images to smaller resolution |

**Recommended image setup for face reconstruction:**
- 8–20 images
- Consistent, diffuse lighting (avoid harsh shadows)
- Neutral background
- Cover: front, left 45°, left 90°, right 45°, right 90°, up 30°, down 30°